In [1]:
import yaml
import numpy as np
import rioxarray
from pathlib import Path

# load config
config_path = Path("..") / "config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# load the three Sentinel-2 bands
s2_dir = Path("..") / "data" / "raw" / "sentinel2"

b03 = rioxarray.open_rasterio(s2_dir / "B03.tif", masked=True)
b04 = rioxarray.open_rasterio(s2_dir / "B04.tif", masked=True)
b08 = rioxarray.open_rasterio(s2_dir / "B08.tif", masked=True)

print(f"B03 shape: {b03.shape}, CRS: {b03.rio.crs}")
print(f"B04 shape: {b04.shape}, CRS: {b04.rio.crs}")
print(f"B08 shape: {b08.shape}, CRS: {b08.rio.crs}")


B03 shape: (1, 4591, 3384), CRS: EPSG:32634
B04 shape: (1, 4591, 3384), CRS: EPSG:32634
B08 shape: (1, 4591, 3384), CRS: EPSG:32634


In [2]:
# NDVI = (NIR - Red) / (NIR + Red)
# squeeze() removes the band dimension so we go from (1, rows, cols) to (rows, cols)
nir = b08.squeeze()
red = b04.squeeze()

ndvi = (nir - red) / (nir + red)

print(f"NDVI shape: {ndvi.shape}")
print(f"NDVI min: {float(ndvi.min()):.3f}")
print(f"NDVI max: {float(ndvi.max()):.3f}")
print(f"NDVI mean: {float(ndvi.mean()):.3f}")


NDVI shape: (4591, 3384)
NDVI min: -0.710
NDVI max: 0.837
NDVI mean: 0.329


In [3]:
# save NDVI to processed/
processed_dir = Path("..") / "data" / "processed"
processed_dir.mkdir(exist_ok=True)

ndvi.rio.to_raster(processed_dir / "ndvi.tif")
print("Saved ndvi.tif")


Saved ndvi.tif


In [4]:
# NDWI = (Green - NIR) / (Green + NIR)
green = b03.squeeze()

ndwi = (green - nir) / (green + nir)

print(f"NDWI min: {float(ndwi.min()):.3f}")
print(f"NDWI max: {float(ndwi.max()):.3f}")
print(f"NDWI mean: {float(ndwi.mean()):.3f}")


NDWI min: -0.809
NDWI max: 0.568
NDWI mean: -0.299


In [5]:
ndwi.rio.to_raster(processed_dir / "ndwi.tif")
print("Saved ndwi.tif")


Saved ndwi.tif


In [6]:
# load DEM
dem_path = Path("..") / "data" / "raw" / "dem" / "dem_30m.tif"
dem = rioxarray.open_rasterio(dem_path, masked=True).squeeze()

print(f"DEM shape: {dem.shape}")
print(f"DEM CRS: {dem.rio.crs}")

# reproject to metric CRS for slope calculation
dem_metric = dem.rio.reproject("EPSG:3067")
res = dem_metric.rio.resolution()

print(f"Reprojected CRS: {dem_metric.rio.crs}")
print(f"Resolution after reproject: {res}")  # should be ~30m


DEM shape: (1442, 1081)
DEM CRS: EPSG:4326
Reprojected CRS: EPSG:3067
Resolution after reproject: (30.467874916727236, -30.467874916727236)


In [9]:
pixel_size = abs(res[0])

# fill NaN with mean elevation before gradient — prevents NaN propagation
elev_filled = np.where(np.isnan(elevation), np.nanmean(elevation), elevation)

dzdx = np.gradient(elev_filled, pixel_size, axis=1)
dzdy = np.gradient(elev_filled, pixel_size, axis=0)

slope_arr = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))

# restore NaN mask where original elevation was missing
slope_arr = np.where(np.isnan(elevation), np.nan, slope_arr)

print(f"Slope min: {np.nanmin(slope_arr):.2f}°")
print(f"Slope max: {np.nanmax(slope_arr):.2f}°")
print(f"Slope mean: {np.nanmean(slope_arr):.2f}°")


Slope min: 0.00°
Slope max: 73.06°
Slope mean: 4.37°


In [11]:
# wrap slope array back into xarray DataArray using dem_metric as spatial template
slope = dem_metric.copy(data=slope_arr)
slope.attrs["long_name"] = "slope_degrees"

# save in EPSG:3067 — we'll align with Sentinel-2 grid in Phase 3
slope.rio.to_raster(processed_dir / "slope.tif")
print(f"Saved slope.tif  (CRS: {slope.rio.crs})")


Saved slope.tif  (CRS: EPSG:3067)
